In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# pip install scikit-learn
# pip install openpyxl
from sklearn.decomposition import PCA
from numpy.linalg import inv
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import os

In [ ]:
os.chdir("/Users/yanni_zheng/Library/CloudStorage/OneDrive-ColumbiaBusinessSchool/classes/25 spring/multi-asset/project/0606")


Universe Choices

In [75]:
#Universe Construction
#Shifting path and following i will all use relative path
selected_universes = ["Global Developed"]

universe_map = pd.read_excel("./data/specs.xlsx",sheet_name="Universe Choices")

selected_tickers_info = universe_map[universe_map["Universe"].isin(selected_universes)]

for universe in selected_universes:

    print(f"\n🔄 Processing Universe: {universe}")

    # === bbg2name mapping ===
    universe_spec = universe_map[universe_map["Universe"] == universe]
    bbg2name = dict(zip(universe_spec["Ticker"], universe_spec["Name"]))
    tickers = universe_spec["Ticker"].tolist()

    # === Try to read index_returns.pkl ===
    ir_pkl_path = Path(f"./data/{universe}_index_returns.pkl")

    if ir_pkl_path.exists():
        print(f"✅ Found existing {universe}_index_returns.pkl, loading directly.")
        index_returns = pd.read_pickle(ir_pkl_path)
    else:
        # === Fall back to loading original data ===
        csv_path = Path(f"./data/{universe}_data.csv")

        index_data = pd.read_csv(csv_path, parse_dates=True)
        #index_data.columns = index_data.columns.str.strip()
        index_data = index_data.rename(columns=bbg2name)
        
        if 'Date' in index_data.columns:
            index_data['Date'] = pd.to_datetime(index_data['Date'])
            index_data.set_index('Date', inplace=True)

        index_returns = index_data.ffill().pct_change().dropna()
    
        # Save for future use
        index_returns.to_pickle(ir_pkl_path)
        print(f"💾 Saved index_returns to {ir_pkl_path}")

    # === Filter columns ===
    valid_columns = [bbg2name[t] for t in bbg2name if bbg2name[t] in index_returns.columns]
    returns_clean = index_returns[valid_columns].dropna()

    # === VIF Analysis ===
    vif_data = pd.DataFrame()
    vif_data["Asset"] = returns_clean.columns
    vif_data["VIF"] = [variance_inflation_factor(returns_clean.values, i) for i in range(returns_clean.shape[1])]

    print("\n📊 === Variance Inflation Factor (VIF) Results ===")
    print(vif_data.sort_values(by="VIF", ascending=False))
    # === Run decomposition ===
    r_fixed, r_random = run_sensitivity_analysis(
        index_returns,
        gamma=0.01,
        shrinkage_vec=[0, 0.2, 0.4, 0.6, 0.8],
        required_variation_explained=np.arange(0, 1.1, 0.1),
        time_windows=[10, 20]
    )

    # Add universe name
    r_fixed["Universe"] = universe
    r_random["Universe"] = universe

    # Save results
    r_fixed.to_csv(f"./result/{universe}_PCA.csv", index=False)
    r_random.to_csv(f"./result/{universe}_Shrinkage.csv", index=False)

  


🔄 Processing Universe: Global Developed
✅ Found existing Global Developed_index_returns.pkl, loading directly.

📊 === Variance Inflation Factor (VIF) Results ===
                 Asset        VIF
20        US Corp Bond  29.929214
19         US Gov Bond  24.722022
13       UK  Corp Bond  21.594387
9    Canada Corp Bond   21.451252
8     Canada Gov Bond   18.079062
12         UK Gov Bond  16.510262
11       EMU Corp Bond  16.099339
16  Emerging Corp Bond  13.534783
17   Emerging Gov Bond  11.372197
10        EMU Gov Bond   9.692642
21       US High Yield   8.524724
22              US MBS   7.560068
6          US S&P 500    6.981738
15     Japan Corp Bond   6.067179
5        CA S&P/TSX 60   5.975531
14      Japan Gov Bond   5.322615
1        EMU Stoxx 50    4.856386
18         AU Gov Bond   4.193417
4         UK FTSE 100    3.881842
0      AU S&P/ASX 200    3.694997
2        Emerging MSCI   3.442593
3    Japan Nikkei 225    3.174494
7            US REITs    2.824704
q 1.0
n_fac 0
q 0.841

In [ ]:
def invpd(Sigma):
    """
    Computes the inverse of a positive definite matrix Sigma.
    If Sigma is not positive definite, it raises an error.
    """
    L = np.linalg.cholesky(Sigma)

    # Compute the inverse using the Cholesky factor
    return np.linalg.inv(L.T) @ np.linalg.inv(L)
def simulate_sensitivity_model(index_returns, shrinkage_lambda=0,annualization_factor = 12 , gamma=0.01, n_sim=1000):
    T = 100 # index_returns.shape[0]
    n_assets = index_returns.shape[1]

    # Estimated covariance (annualized)
    Sigma = index_returns.cov().values * annualization_factor 
    vol = np.sqrt(np.diag(Sigma))
    vol_df = pd.DataFrame({
        'Asset': index_returns.columns,
        'Volatility': vol
    })
    
    # print("Vol", vol_df)
    # Apply shrinkage if needed
    if shrinkage_lambda > 0:
        # shrink_target = np.identity(n_assets) * np.trace(Sigma) / n_assets
        shrink_target = np.diag(np.diag(Sigma))
        Sigma = (1 - shrinkage_lambda) * Sigma + shrinkage_lambda * shrink_target


    # Omega_0 = covariance of theta
    Omega_0 = Sigma / T
    theta_0 = np.zeros((n_assets, 1))  # mean zero
    return MLBsim(Sigma,Omega_0,theta_0,gamma,n_sim)

def MLBsim(Sigma,Omega_0,theta_0,gamma,n_sim):
    # M² = trace(Sigma⁻¹) / n
    n_assets = Sigma.shape[0]
    Sigma_inv = np.linalg.inv(Sigma)
    trace_Sigma_inv = np.trace(Sigma_inv)
    if trace_Sigma_inv<0:
        print("Warning: trace_Sigma_inv is negative, check your covariance matrix Sigma.")
    M = np.sqrt(trace_Sigma_inv / n_assets)
    if pd.isna(M):
        print("M is NaN, check your covariance matrix Sigma.")
    # L = 1 because Γ = Omega is fixed
    L = 1.0

    # Simulate theta and compute h
    h_1_norm_list = []
    h_2_norm_list = []

    for _ in range(n_sim):
        theta = np.random.multivariate_normal(mean=np.zeros(n_assets), cov=Omega_0).reshape(-1, 1)
        h = gamma * Sigma_inv @ theta
        h_1 = np.sum(np.abs(h))
        h_2 = np.sum(h ** 2)
        h_1_norm_list.append(h_1)
        h_2_norm_list.append(h_2)

    # Compute expectation estimates
    E_l1_squared = (np.mean(h_1_norm_list)) ** 2
    E_l2 = np.mean(h_2_norm_list)

    B = E_l1_squared / E_l2
    S = M * L * np.sqrt(B)

    return { "M": M, "L": L,"B": B, "S": S}

def simulate_sensitivity_pca_model(index_returns,annualization_factor=12, required_var_explained=None, gamma=0.01, n_sim=1000):
    tau=1/100
    returns_matrix = index_returns.values
    len(np.argwhere(np.isnan(returns_matrix)))
    Sigma = np.cov(returns_matrix.T) * annualization_factor 
    theta_0 = np.zeros((n_assets, 1))  # mean zero
    # === Step 1: PCA Decomposition ===
    # n_factors = min(n_factors, T, n_assets)
    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
    reverse = np.arange(n_assets,0,-1)-1
    eigenvalues=eigenvalues[reverse]
    eigenvectors=eigenvectors[:,reverse]
    totalvar=eigenvalues.sum()
    varexplained = eigenvalues.cumsum()/totalvar
    n_fac=np.argmax(varexplained>=required_var_explained)
    q=(varexplained[n_fac]-required_var_explained)/(eigenvalues[n_fac]/totalvar) 
    print('q', q)
    print('n_fac', n_fac)
    if q>1 or q<0:
        raise Exception('q must be in the range [0,1]') # q is the ratio of the explained variance of the n_factors to the total variance
# print('n_fac', n_fac)
    # print('n_fac', n_fac)
    if n_fac==0:
        Sigma_pca=np.zeros((n_assets,n_assets))
    else:
        Sigma_pca = eigenvectors[:,:n_fac] @ np.diag(eigenvalues[:n_fac]) @ eigenvectors[:,:n_fac].T
    if q>=0: # we use only a fraction q of the last principal component
        # Ensure eigenvectors[:, n_fac] is treated as a 2D column vector
        v = np.atleast_2d(eigenvectors[:, n_fac]).T  # Convert to column vector
        Sigma_pca = Sigma_pca + ((1 - q)) * eigenvalues[n_fac] * (v @ v.T)  # Outer product
    else:
        raise Exception('impossible: q cannot be negative')
    # try:
    #     invpd(Sigma_pca)
    # raise Exception("Matrix is singular.")
    if any(np.diag(Sigma)<np.diag(Sigma_pca)):
        pd.DataFrame({'sig':np.diag(Sigma).T,'pca':np.diag(Sigma_pca).T})
        print("Warning: PCA covariance matrix has larger diagonal elements than original covariance matrix.")
    np.fill_diagonal(Sigma_pca, np.diag(Sigma))
    calMsq =invpd(Sigma_pca).trace()/n_assets
    if calMsq<0:
        print("Warning: M² ]is negative, check your covariance matrix Sigma.")
        # raise Exception('impossible: PCA covariance matrix should not be singular') 
    # except np.linalg.LinAlgError as e:
    #     print("Matrix is singular. Exception details:", e)
    Sigma_pca[-1,-1]-Sigma[-1,-1]
    Omega_0 = tau*Sigma_pca

    # explained_variance_ratio= np.sum(evaln) / np.sum(eigenvalues)  # Explained variance ratio
    mlb= MLBsim(Sigma_pca,Omega_0,theta_0,gamma,n_sim)
     
    mlb.update({"explained_variance_ratio":required_var_explained, "n_factors": n_fac})
    return mlb

def simulate_sensitivity_pca_model_old(index_returns,annualization_factor=12, n_factors=None, gamma=0.01, n_sim=1000):
    n_returns, n_assets = index_returns.shape
    T=100
    returns_matrix = index_returns.values
    len(np.argwhere(np.isnan(returns_matrix)))
    Sigma = np.cov(returns_matrix.T) * annualization_factor 
    theta_0 = np.zeros((n_assets, 1))  # mean zero
    # === Step 1: PCA Decomposition ===
    n_factors = min(n_factors, T, n_assets)
    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
    evecn=eigenvectors[:,-n_factors:]
    evaln=eigenvalues[-n_factors:]
    Sigma_pca=evecn@np.diag(evaln)@evecn.T
    np.fill_diagonal(Sigma_pca, np.diag(Sigma))
    explained_variance_ratio= np.sum(evaln) / np.sum(eigenvalues)  # Explained variance ratio
    Omega_0= Sigma_pca / T
    mlb= MLBsim(Sigma_pca,Omega_0,theta_0,gamma,n_sim)
     
    mlb.update({"explained_variance_ratio":explained_variance_ratio, "n_factors": n_factors})
    return mlb
    

def run_sensitivity_analysis(index_returns, gamma, shrinkage_vec,required_variation_explained,time_windows):
    results_fixed = []
    results_random = []
    for window in time_windows:
        end_date = index_returns.index.max()
        start_date = end_date - pd.DateOffset(years=window)
        data_slice = index_returns[start_date:]
        for i in np.arange(len(shrinkage_vec)):
            result_random = simulate_sensitivity_model(data_slice,  gamma=gamma, shrinkage_lambda=shrinkage_vec[i])
            result_random.update({"Shrinkage Lambda": shrinkage_vec[i],"Time Window (Years)": window})
            results_random.append(result_random)
        for i in np.arange(len(required_variation_explained)):
            result_fixed = simulate_sensitivity_pca_model(data_slice,  required_var_explained=required_variation_explained[i],gamma=gamma)
            result_fixed.update({"Time Window (Years)": window})
            results_fixed.append(result_fixed)         
    results_df_fixed = pd.DataFrame(results_fixed)
    print(results_df_fixed)
    results_df_fixed.to_csv(f"sensitivity_summary_generalized_PCA_{100*gamma}%.csv", index=False)
    
    results_df_random = pd.DataFrame(results_random)
    results_df_random.to_csv(f"sensitivity_summary_generalized_random_{100*gamma}%.csv", index=False)
    return results_df_fixed,results_df_random

Characteristics Experiements

In [ ]:
# As I don't have access to bbg api， i create the mapping dictionary by myself and save it in the org_data1 sheet  2
## Load and Preprocess Data from 2004-2024; from six Russell and Bloomberg Barclays bond indices
index_data = pd.read_excel("data/org_data0604.xlsx",sheet_name = "Index")
#index_data.drop('TX60AR Index',axis=1,inplace=True) # colm change 1
# clean up blanks
fixblanks={x:x.replace("\u202f"," ") for x in list(index_data.columns)}
#yanni 0530
bbg2name = {
    "AS51 Index"     : "AU S&P/ASX 200",
    "SX5E Index"     : "EMU Stoxx 50",
    "MXEF Index"     : "Emerging MSCI",
    "NKY Index"      : "Japan Nikkei 225",
    "UKX Index"      : "UK FTSE 100",
    "TX60AR Index"   : "CA S&P/TSX 60",
    "SPX Index"      : "US S&P 500",
    "FNRET Index"    : "US REITs",
    "I05500CA Index" : "Canada Gov Bond",
    "I05510CA Index" : "Canada Corp Bond",
    "LEATTREU Index" : "EMU Gov Bond",
    "LECPTREU Index" : "EMU Corp Bond",
    "LSG1TRGU Index" : "UK Gov Bond",
    "LC61TRGU Index" : "UK Corp Bond",
    "I38292JP Index" : "Japan Gov Bond",
    "SPBJPCPT Index" : "Japan Corp Bond",
    "I12877US Index" : "Emerging Corp Bond",
    "BEMUTRUU Index" : "Emerging Gov Bond",
    "BATY0 Index"    : "AU Gov Bond",
    "BACR0 Index"    : "AU Corp Bond",
    "LUATTRUU Index" : "US Gov Bond",
    "LUACTRUU Index" : "US Corp Bond",
    "LF98TRUU Index" : "US High Yield",
    "LUMSTRUU Index" : "US MBS"
}

index_data=index_data.rename(columns=fixblanks).rename(columns=bbg2name) # colm 1
index_data=index_data.rename(columns=fixblanks)
list(bbg2name[x] if x in bbg2name.keys() else 'not found' for x in list(index_data.columns))
#ndex_data.columns.sort_values()
#sorted(list(bbg2name.keys()))
#set(index_data.columns)-set(bbg2name.keys())
index_data['Date'] = pd.to_datetime(index_data['Date'])
index_data.set_index('Date', inplace=True)
index_returns = index_data.fillna(method='ffill').pct_change().dropna() # colm change 2
returns_clean = index_returns.dropna()
index_returns.to_csv('data/org_data_copy.csv') # colm 3

# Save index_returns to a pickle file
index_returns.to_pickle('data/rename_index_returns.pkl') # Save to pickle


# Test the collinearity

vif_data = pd.DataFrame()
vif_data["Asset"] = returns_clean.columns
vif_data["VIF"] = [variance_inflation_factor(returns_clean.values, i) for i in range(returns_clean.shape[1])]

print(vif_data)
print(index_returns.head(1))


,AU S&P/ASX 200,EMU Stoxx 50,Emerging MSCI,Japan Nikkei 225,UK FTSE 100,CA S&P/TSX 60,US S&P 500,US REITs,Canada Gov Bond,Canada Corp Bond,...,Japan Gov Bond,Japan Corp Bond,Emerging Corp Bond,Emerging Gov Bond,AU Gov Bond,AU Corp Bond,US Gov Bond,US Corp Bond,US High Yield,US MBS
Date,,,,,,,,,,,,,,,,,,,,,
2004-01-30,3271.997,2839.13,457.19,10783.61,4390.70,889.89,1131.13,NaN,108.8665,112.2252,...,NaN,87.052,122.5498,NaN,4462.054,4081.851,1338.69,1447.76,705.47,1190.05
2004-02-27,3360.598,2893.18,477.73,11041.92,4492.20,916.98,1144.94,NaN,110.3387,113.8025,...,NaN,87.321,123.1676,NaN,4526.031,4121.945,1355.28,1465.79,703.70,1200.08
2004-03-31,3415.258,2787.49,482.06,11715.39,4385.70,891.01,1126.21,NaN,110.9268,114.2131,...,NaN,86.950,125.9259,NaN,4566.396,4163.212,1368.00,1479.63,708.48,1205.34
2004-04-30,3400.793,2787.48,441.30,11761.79,4489.70,854.99,1107.31,NaN,109.5991,113.0920,...,NaN,86.841,121.4708,NaN,4513.852,4138.786,1324.00,1433.11,703.66,1183.88
2004-05-31,3460.210,2749.62,431.26,11236.37,4430.70,870.95,1120.68,NaN,109.0068,112.2397,...,NaN,86.993,120.9967,NaN,4557.376,4174.720,1319.45,1422.74,691.74,1181.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-31,8532.300,5286.87,1093.37,39572.49,8673.96,5218.44,6040.53,268.65,208.6989,273.6613,...,103.6298,102.834,301.0339,146.0846,10354.774,11519.210,2302.09,3307.68,2719.80,2166.52
2025-02-28,8172.354,5463.54,1097.25,37155.50,8809.74,5197.22,5954.50,278.62,211.0117,275.6553,...,102.8398,102.514,306.3407,148.3769,10447.957,11610.393,2351.72,3375.11,2738.06,2221.76
2025-03-31,7843.424,5248.39,1101.40,35617.56,8582.81,5096.08,5611.85,267.39,210.6698,275.6646,...,101.8035,102.204,306.5031,146.6052,10460.154,11643.402,2357.12,3365.38,2710.07,2221.38


In [ ]:
#rename the columns of the worksheets in characteristics excel
char_path = "data/characteristics.xlsx"

bbg2name = {
    "AS51 Index": "AU S&P/ASX 200",
    "SX5E Index": "EMU Stoxx 50",
    "MXEF Index": "Emerging MSCI",
    "NKY Index": "Japan Nikkei 225",
    "UKX Index": "UK FTSE 100",
    "TX60AR Index": "CA S&P/TSX 60",
    "SPX Index": "US S&P 500",
    "FNRET Index": "US REITs",
    "I05500CA Index": "Canada Gov Bond",
    "I05510CA Index": "Canada Corp Bond",
    "LEATTREU Index": "EMU Gov Bond",
    "LECPTREU Index": "EMU Corp Bond",
    "LSG1TRGU Index": "UK Gov Bond",
    "LC61TRGU Index": "UK Corp Bond",
    "I38292JP Index": "Japan Gov Bond",
    "SPBJPCPT Index": "Japan Corp Bond",
    "I12877US Index": "Emerging Corp Bond",
    "BEMUTRUU Index": "Emerging Gov Bond",
    "BATY0 Index": "AU Gov Bond",
    "BACR0 Index": "AU Corp Bond",
    "LUATTRUU Index": "US Gov Bond",
    "LUACTRUU Index": "US Corp Bond",
    "LF98TRUU Index": "US High Yield",
    "LUMSTRUU Index": "US MBS"
}

xls = pd.ExcelFile(char_path)
sheet_names = xls.sheet_names

with pd.ExcelWriter("data/characteristics_renamed.xlsx", engine="openpyxl") as writer:
    for sheet in sheet_names:
        df = pd.read_excel(char_path, sheet_name=sheet, index_col=0, parse_dates=True)

        if "Equity" in sheet or "Bond" in sheet:
            df.rename(columns=bbg2name, inplace=True)

        df.to_excel(writer, sheet_name=sheet)


/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/795943904.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_excel(char_path, sheet_name=sheet, index_col=0, parse_dates=True)


In [ ]:
# generate all characteristics we need
characteristics_path = "data/characteristics_renamed.xlsx"

# Compute 1-Year Momentum: (P_t-1 / P_t-12) - 1
momentum_1yr = index_data.shift(1) / index_data.shift(12) - 1
momentum_1yr.name = "1yr Momentum"

# Compute 1-Year Reversal: return from t-12 to t-11
monthly_returns = index_data.pct_change()
reversal_1yr = monthly_returns.shift(12)
reversal_1yr.name = "1yr Reversal"

constant_1 = pd.DataFrame(1, index=index_data.index, columns=index_data.columns)
constant_1.name = "Constant_1"

# 2) Bond indicator
indicator_bond = pd.DataFrame(
    [[1 if "Bond" in col else 0 for col in index_data.columns]],
    index=[index_data.index[0]],
    columns=index_data.columns
).reindex(index=index_data.index, method='ffill')


# Write to Excel if sheets do not exist
# Check existing sheets first — do this OUTSIDE the writer block
with pd.ExcelFile(characteristics_path) as reader:
    existing_sheets = reader.sheet_names

# Now safely write
with pd.ExcelWriter(characteristics_path, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
    if "1yr Momentum" not in existing_sheets:
        momentum_1yr.to_excel(writer, sheet_name="1yr Momentum")
    if "1yr Reversal" not in existing_sheets:
        reversal_1yr.to_excel(writer, sheet_name="1yr Reversal")
    if "Constant_1" not in existing_sheets:
        constant_1.to_excel(writer, sheet_name="Constant_1")
    if "Indicator_Bond" not in existing_sheets:
        indicator_bond.to_excel(writer, sheet_name="Indicator_Bond")
    



In [ ]:
char_file = pd.ExcelFile("data/characteristics_renamed.xlsx")
sub_returns_clean = pd.read_pickle("data/rename_index_returns.pkl")
char_sheets = char_file.sheet_names
target_sheets = ["Equity PE", "Equity Market Cap", "Bond Yield", "Bond OAS", "1yr Momentum", "1yr Reversal", "Constant_1", "Indicator_Bond"]
tau = 1 / 100  # small scalar for Omega prior
annualization_factor = 12
gamma = 0.01


char_results = []
for sheet in target_sheets:
    print("sheet name is", sheet)
    if sheet not in char_sheets:
        continue

    char_df = pd.read_excel("data/characteristics_renamed.xlsx", sheet_name=sheet, index_col=0, parse_dates=True)
    results = []

    for date in char_df.index:
        if date not in char_df.index or date not in sub_returns_clean.index:
            continue

        theta_raw = char_df.loc[date]
        assets = theta_raw.index.intersection(sub_returns_clean.columns)
        theta_raw = theta_raw[assets]


        # if all values are missing, skip this date
        if theta_raw.isna().all():
            continue

        # if some values are missing, fill them with N(0, sigma^2)
        returns_sub = sub_returns_clean[assets].dropna()
        #returns_sub = returns_sub[returns_sub.index <= date].dropna()
        if returns_sub.shape[0] < 12:
            continue

        Sigma = returns_sub.cov().values * annualization_factor

        missing = theta_raw.isna()
        if missing.any():
            std_dev = np.sqrt(np.diag(Sigma))
            theta_raw[missing] = np.random.normal(loc=0, scale=std_dev[missing.values])
        if len(assets) < 3:
            print(f"{sheet} - {date}: number of valid assets = {len(assets)}")
            continue

        try:
            Sigma_inv = np.linalg.inv(Sigma)
        except np.linalg.LinAlgError:
            continue

        theta = StandardScaler().fit_transform(theta_raw.values.reshape(-1, 1))

        h = gamma * Sigma_inv @ theta
        h1 = np.sum(np.abs(h))
        h2 = np.sum(h ** 2)
        if h2 == 0:
            continue
        B = (h1 ** 2) / h2

        Omega_diag = np.diag(np.diag(Sigma))
        try:
            Omega_inv = np.linalg.inv(tau * Omega_diag)
        except np.linalg.LinAlgError:
            continue

        numerator = float(theta.T @ Sigma_inv @ theta)
        denominator = float(theta.T @ Omega_inv @ theta)
        if denominator <= 0:
            continue
        L = np.sqrt(numerator / denominator)

        M = np.sqrt(np.trace(Sigma_inv) / len(assets))
        S = M * L * np.sqrt(B)

        results.append({"Date": date, "M": M, "B": B, "L": L, "S": S})

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        avg_values = results_df[["M", "B", "L", "S"]].mean()
        avg_values["Characteristic"] = sheet
        char_results.append(avg_values)

char_results_df = pd.DataFrame(char_results).set_index("Characteristic")
print(char_results_df.round(4))

sheet name is Equity PE
sheet name is Equity Market Cap


/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  numerator = float(theta.T @ Sigma_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  denominator = float(theta.T @ Omega_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated N

sheet name is Bond Yield


/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  numerator = float(theta.T @ Sigma_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  denominator = float(theta.T @ Omega_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated N

sheet name is Bond OAS


...

/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  numerator = float(theta.T @ Sigma_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  denominator = float(theta.T @ Omega_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated N

sheet name is 1yr Reversal


/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  numerator = float(theta.T @ Sigma_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  denominator = float(theta.T @ Omega_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated N

sheet name is Constant_1
sheet name is Indicator_Bond
                          M       B       L        S
Characteristic                                      
Equity PE           13.3338  4.9869  0.1686   4.9995
Equity Market Cap   13.3338  3.9400  0.1454   3.8491
Bond Yield         103.6413  3.6340  0.1682  33.2016
Bond OAS            94.6731  3.5534  0.1592  28.4100
1yr Momentum        88.1995  4.2484  0.1594  28.3152
1yr Reversal        88.1995  5.2926  0.1667  33.4206
Indicator_Bond      88.1995  6.1681  0.1759  38.5232


/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  numerator = float(theta.T @ Sigma_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:70: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  denominator = float(theta.T @ Omega_inv @ theta)
/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_15637/4018264132.py:69: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated N

In [41]:
char_results_df

,M,B,L,S
Characteristic,,,,
Equity PE,13.333812,4.986934,0.168607,4.999474
Equity Market Cap,13.333812,3.940006,0.145430,3.849063
Bond Yield,103.641270,3.634005,0.168209,33.201566
Bond OAS,94.673106,3.553372,0.159151,28.410000
1yr Momentum,88.199462,4.248405,0.159360,28.315185
1yr Reversal,88.199462,5.292580,0.166740,33.420610
Indicator_Bond,88.199462,6.168093,0.175866,38.523221
